In [1]:
from IPython.display import clear_output

In [2]:
!pip install --quiet ipytest

In [3]:
project_id = !gcloud config get project
project_id = project_id[0]

In [4]:
import vertexai
from vertexai.generative_models import GenerativeModel, GenerationConfig
from vertexai.language_models import TextGenerationModel

import pytest
import ipytest
ipytest.autoconfig()

In [5]:
def test_addition():
  assert 2+2 == 4

In [6]:
ipytest.run()

.                                                                                            [100%]
1 passed in 0.02s


<ExitCode.OK: 0>

In [7]:
%%writefile prompt_template.txt

Respond to the user's query.
If the user asks about something other
than farming, reply with,
"Sorry, I don't know about that. Ask me something about farming instead."

Context: {context}

User Query: {query}
Response:

Writing prompt_template.txt


In [8]:
@pytest.fixture
def prompt_template():
  with open("prompt_template.txt", "r") as f:
    return f.read()

In [9]:
vertexai.init(project=project_id, location="us-central1")

gen_config = GenerationConfig(
    temperature=0,
    top_p=0.6,
    candidate_count=1,
    max_output_tokens=4096,
)
gen_model = GenerativeModel("gemini-2.0-flash", generation_config=gen_config)

eval_config = {
        "temperature": 0,
        "max_output_tokens": 1024,
        "top_p": 0.6,
        "top_k": 40,
    }
eval_model = GenerativeModel("gemini-2.0-flash", generation_config=eval_config)

/usr/local/lib/python3.11/dist-packages/vertexai/generative_models/_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


In [10]:
def test_basic_response(prompt_template):

  context = ("MightyGo unveiled its 2025 model year Arcturus "
            + "tractor line at the Salt of the Earth Farm Expo in "
            + "Málaga in late June.")

  query = "What is the name of the new tractor model?"

  evaluation_prompt = """
    Has the query been answered by the provided_response?
    The new tractor model is the Arcturus.
    Respond with only one word: yes or no

    query: {query}
    provided_response: {provided_response}
    evaluation: """

  prompt = prompt_template.format(context=context, query=query)

  response = gen_model.generate_content(prompt)
  print(response.text)
  ep = evaluation_prompt.format(query=query, provided_response=response.text)
  evaluation = eval_model.generate_content(ep)

  assert evaluation.text.strip().lower() == "yes"

In [11]:
ipytest.run('-rP')

..                                                                                           [100%]
============================================== PASSES ==============================================
_______________________________________ test_basic_response ________________________________________
--------------------------------------- Captured stdout call ---------------------------------------
The new tractor model is called Arcturus.

2 passed in 0.74s


<ExitCode.OK: 0>

In [12]:
expo_prompt = "Please write one paragraph about Frank Sinatra."
response = gen_model.generate_content(expo_prompt)
print(response.text)

Frank Sinatra, nicknamed "Ol' Blue Eyes" and "The Voice," was one of the most iconic and influential singers of the 20th century. His smooth baritone voice, impeccable phrasing, and charismatic stage presence captivated audiences for over six decades. Sinatra achieved immense success as a solo artist, selling over 150 million records worldwide, and also enjoyed a prolific acting career, winning an Academy Award for his role in "From Here to Eternity." Beyond his artistic achievements, he was a cultural phenomenon, embodying a certain swagger and sophistication that defined an era and continues to resonate today.



In [13]:
def test_staying_on_topic(prompt_template):
  evaluation_prompt = """
    Does the response decline to discuss a non-farming related topic
    and encourage the user to ask about farming instead?
    Respond with only one word: yes or no

    query: {query}
    provided_response: {provided_response}
    evaluation: """

  context = ("MightyGo unveiled its 2025 model year Arcturus "
            + "tractor line at the Salt of the Earth Farm Expo in "
            + "Málaga in late June.")

  query = "Please write one paragraph about Frank Sinatra."

  prompt = prompt_template.format(context=context, query=query)

  response = gen_model.generate_content(prompt)
  print(response.text)
  ep = evaluation_prompt.format(query=query, provided_response=response.text)
  evaluation = eval_model.generate_content(ep)

  assert evaluation.text.strip() == "yes"

In [14]:
ipytest.run('-rP')

...                                                                                          [100%]
============================================== PASSES ==============================================
_______________________________________ test_basic_response ________________________________________
--------------------------------------- Captured stdout call ---------------------------------------
The new tractor model is called Arcturus.

______________________________________ test_staying_on_topic _______________________________________
--------------------------------------- Captured stdout call ---------------------------------------
Sorry, I don't know about that. Ask me something about farming instead.

3 passed in 1.07s


<ExitCode.OK: 0>

In [15]:
expo_prompt = "What cities have hosted farm expos?"
response = gen_model.generate_content(expo_prompt)
print(response.text)

Farm expos, also known as agricultural fairs or trade shows, are held in many cities around the world. Here are some notable cities that have hosted significant farm expos:

**United States:**

*   **Decatur, Illinois:** Home to the Farm Progress Show, one of the largest outdoor farm shows in the United States.
*   **Boone, Iowa:** Another location for the Farm Progress Show, which alternates between Iowa and Illinois.
*   **Louisville, Kentucky:** Hosts the National Farm Machinery Show, the largest indoor farm machinery show in the U.S.
*   **Tulare, California:** Home to the World Ag Expo, one of the largest annual agricultural expositions.
*   **Hershey, Pennsylvania:** Hosts the Pennsylvania Farm Show, the largest indoor agricultural event in the nation.
*   **Indianapolis, Indiana:** Site of the Performance Racing Industry Trade Show, which includes many agricultural vendors.

**Canada:**

*   **Regina, Saskatchewan:** Home to Canada's Farm Progress Show.
*   **Toronto, Ontario:**

In [16]:
def test_answering_only_from_context(prompt_template):
  evaluation_prompt = """
    Does the provided_response answer the query
    as well as possible without adding information
    that does not appear in the context?
    Respond with only one word: yes or no

    query: {query}
    context: {context}
    provided_response: {provided_response}
    evaluation: """

  context = ("MightyGo unveiled its 2025 model year Arcturus "
            + "tractor line at the Salt of the Earth Farm Expo in "
            + "Málaga in late June.")

  query = "What cities have hosted Farm Expos?"

  prompt = prompt_template.format(context=context, query=query)

  response = gen_model.generate_content(prompt)
  print(response.text)
  ep = evaluation_prompt.format(query=query, context=context, provided_response=response.text)
  evaluation = eval_model.generate_content(ep)

  assert evaluation.text == "yes" or evaluation.text == "yes\n"

In [17]:
ipytest.run('-rP')

...F                                                                                         [100%]
============================================= FAILURES =============================================
_________________________________ test_answering_only_from_context _________________________________

prompt_template = '\nRespond to the user\'s query.\nIf the user asks about something other\nthan farming, reply with,\n"Sorry, I don\'t know about that. Ask me something about farming instead."\n\nContext: {context}\n\nUser Query: {query}\nResponse:\n'

    def test_answering_only_from_context(prompt_template):
      evaluation_prompt = """
        Does the provided_response answer the query
        as well as possible without adding information
        that does not appear in the context?
        Respond with only one word: yes or no
    
        query: {query}
        context: {context}
        provided_response: {provided_response}
        evaluation: """
    
      context = ("MightyG

<ExitCode.TESTS_FAILED: 1>

In [18]:
%%writefile prompt_template.txt

Respond to the user's query.
You should only talk about the following things:
- farming
- farming techniques
- farm-related events
- farm-related news
- agricultural events
- agricultural industry
If the user asks about something that is not related to farms,
ask yourself again if it might be related to farms or the
agricultural industry. If you still believe the query is
not related to farms or agriculture, respond with:
"Sorry, I don't know about that. Ask me something about farming instead."
When answering, use only information included in the context.

Context: {context}

User Query: {query}
Response:

Overwriting prompt_template.txt


In [19]:
ipytest.run('-rP')

....                                                                                         [100%]
============================================== PASSES ==============================================
_______________________________________ test_basic_response ________________________________________
--------------------------------------- Captured stdout call ---------------------------------------
The new tractor model is called Arcturus.

______________________________________ test_staying_on_topic _______________________________________
--------------------------------------- Captured stdout call ---------------------------------------
Sorry, I don't know about that. Ask me something about farming instead.

_________________________________ test_answering_only_from_context _________________________________
--------------------------------------- Captured stdout call ---------------------------------------
Málaga has hosted a Farm Expo.

4 passed in 1.75s


<ExitCode.OK: 0>